In [1]:
import pandas as pd
import numpy as np

# Mock product dataset
data = [
    {"product_name": "Maggi Noodles", "selling_rate": 0.9, "margin": 20, "inventory_days": 5, "category": "Food", "brand": "branded"},
    {"product_name": "Tea Pack", "selling_rate": 0.85, "margin": 30, "inventory_days": 7, "category": "Beverage", "brand": "branded"},
    {"product_name": "Biscuits", "selling_rate": 0.8, "margin": 25, "inventory_days": 6, "category": "Food", "brand": "branded"},
    {"product_name": "Shampoo", "selling_rate": 0.6, "margin": 80, "inventory_days": 15, "category": "Personal Care", "brand": "branded"},
    {"product_name": "Soap", "selling_rate": 0.7, "margin": 50, "inventory_days": 10, "category": "Personal Care", "brand": "branded"},
    {"product_name": "Face Wash", "selling_rate": 0.5, "margin": 120, "inventory_days": 20, "category": "Personal Care", "brand": "branded"},
    {"product_name": "Store Brand Snacks", "selling_rate": 0.65, "margin": 150, "inventory_days": 12, "category": "Food", "brand": "own"},
    {"product_name": "Store Brand Tea", "selling_rate": 0.6, "margin": 180, "inventory_days": 14, "category": "Beverage", "brand": "own"},
    {"product_name": "Premium Chocolate", "selling_rate": 0.4, "margin": 200, "inventory_days": 25, "category": "Food", "brand": "branded"},
    {"product_name": "Stationary Kit", "selling_rate": 0.3, "margin": 60, "inventory_days": 30, "category": "Stationary", "brand": "branded"}
]

df = pd.DataFrame(data)

# Save to Excel
file_path = "retail_data.xlsx"
df.to_excel(file_path, index=False)

print("Mock dataset created:", file_path)
print(df.head())

Mock dataset created: retail_data.xlsx
    product_name  selling_rate  margin  inventory_days       category    brand
0  Maggi Noodles          0.90      20               5           Food  branded
1       Tea Pack          0.85      30               7       Beverage  branded
2       Biscuits          0.80      25               6           Food  branded
3        Shampoo          0.60      80              15  Personal Care  branded
4           Soap          0.70      50              10  Personal Care  branded


| Scenario                     | Example           |
| ---------------------------- | ----------------- |
| High demand, low margin      | Maggi             |
| Medium demand, medium margin | Soap              |
| Low demand, high margin      | Premium Chocolate |
| Private label                | Store Brand Tea   |
| Slow moving risk             | Stationary        |


In [2]:
# ==============================
# 📦 Imports
# ==============================
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
import pandas as pd
import numpy as np
from scipy.optimize import linprog


# ==============================
# 📥 Tool 1: Load Data
# ==============================
@tool
def load_data(file_path: str):
    """Load retail data from Excel"""
    df = pd.read_excel(file_path)
    return df.to_dict(orient="records")


# ==============================
# 🧮 Tool 2: Feature Engineering
# ==============================
@tool
def compute_features(data: list):
    """Compute profit, cost, and net profit"""
    df = pd.DataFrame(data)

    df["expected_profit"] = df["margin"] * df["selling_rate"]

    # Better holding cost (depends on demand + time)
    df["holding_cost"] = df["inventory_days"] * df["selling_rate"] * 5

    df["net_profit"] = df["expected_profit"] - df["holding_cost"]

    return df.to_dict(orient="records")


# ==============================
# 🧠 Tool 3: Business Rule Adjustments
# ==============================
@tool
def adjust_for_business_rules(data: list):
    """Apply retail heuristics like slow-moving penalty and private label boost"""
    df = pd.DataFrame(data)

    # Penalize slow-moving products
    df.loc[df["selling_rate"] < 0.4, "net_profit"] *= 0.5

    # Boost private label
    df.loc[df["brand"] == "own", "net_profit"] *= 1.2

    # Diversity penalty (avoid same category domination)
    category_counts = df["category"].value_counts()
    df["diversity_penalty"] = df["category"].map(category_counts)
    df["net_profit"] -= df["diversity_penalty"] * 2

    return df.to_dict(orient="records")


# ==============================
# ⚙️ Tool 4: Optimization Engine
# ==============================
@tool
def optimize_shelf(data: list):
    """Optimize shelf allocation using linear programming"""
    df = pd.DataFrame(data)

    c = -df["net_profit"].values  # maximize

    # Constraints:
    # 1. Shelf capacity
    # 2. Warehouse constraint (inventory proxy)
    A = [
        [1] * len(df),
        df["inventory_days"].values
    ]

    b = [20, 200]

    bounds = [(0, 5) for _ in range(len(df))]

    result = linprog(c, A_ub=A, b_ub=b, bounds=bounds)

    if not result.success:
        return {"error": "Optimization failed", "message": result.message}

    df["allocation"] = np.round(result.x, 2)

    return df.to_dict(orient="records")


# ==============================
# 🧾 Tool 5: Generate Recommendation
# ==============================
@tool
def generate_recommendation(data: list):
    """Filter selected products"""
    df = pd.DataFrame(data)

    selected = df[df["allocation"] > 0]

    return selected.to_dict(orient="records")


# ==============================
# 🧠 Tool 6: Explain Decisions
# ==============================
@tool
def explain_decision(data: list):
    """Explain why products were selected"""
    df = pd.DataFrame(data)

    selected = df[df["allocation"] > 0]

    explanations = []

    for _, row in selected.iterrows():
        explanations.append(
            f"{row['product_name']} → allocation {row['allocation']} | net_profit {row['net_profit']:.2f}"
        )

    return explanations


# ==============================
# 🤖 Agent Prompt (Improved)
# ==============================
SYSTEM_PROMPT = """
You are a Retail Optimization Agent.

Your job is to dynamically decide how to optimize shelf placement.

Steps (you can adapt if needed):
1. Load data
2. Compute features
3. Adjust for business rules
4. Optimize shelf allocation
5. Generate recommendation
6. Explain decisions

Focus on:
- Maximizing net profit
- Avoiding slow-moving inventory
- Promoting private label strategically
- Maintaining diversity across categories

You can decide which tools to call and in what order.
"""


# ==============================
# 🤖 Create Agent
# ==============================
llm = ChatOpenAI(model="gpt-4o-mini")

tools = [
    load_data,
    compute_features,
    adjust_for_business_rules,
    optimize_shelf,
    generate_recommendation,
    explain_decision
]

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT
)




In [3]:
# ==============================
# ▶️ Run Agent
# ==============================
response = agent.invoke({
    "messages": [
        ("user", "Optimize shelf using file retail_data.xlsx")
    ]
})

print(response)

{'messages': [HumanMessage(content='Optimize shelf using file retail_data.xlsx', additional_kwargs={}, response_metadata={}, id='e9a78555-0276-40f1-8ae6-8aba831084d9'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_YzRe4ciJYFpFhv6FaJ62w9xO', 'function': {'arguments': '{"file_path":"retail_data.xlsx"}', 'name': 'load_data'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 273, 'total_tokens': 291, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_476b124e6d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--90321ee5-9fa6-4c83-b8b7-174d96bfc964-0', tool_calls=[{'name': 'load_data', 'args': {'file_path': 'retail_data.xlsx'}, 'id': 'call_YzRe4ciJYFpFhv6FaJ62w9xO', 'type':

In [5]:
from IPython.display import Markdown, display
import json

final_message = response["messages"][-1].content

display(Markdown(f"### 📊 Retail Shelf Optimization Result\n\n{final_message}"))

### 📊 Retail Shelf Optimization Result

### Shelf Optimization Recommendations

#### Recommended Allocations:
1. **Store Brand Snacks**
   - **Category:** Food
   - **Brand:** Own
   - **Allocation:** 5.0 units
   - **Net Profit:** $62.20

2. **Store Brand Tea**
   - **Category:** Beverage
   - **Brand:** Own
   - **Allocation:** 5.0 units
   - **Net Profit:** $75.20

3. **Premium Chocolate**
   - **Category:** Food
   - **Brand:** Branded
   - **Allocation:** 2.8 units
   - **Net Profit:** $22.00

### Explanation of Decisions:
- **Store Brand Snacks** and **Store Brand Tea** were prioritized for allocation due to their high net profits of $62.20 and $75.20 respectively. These products also help to strategically promote private label items, which can enhance overall profit margins.
  
- **Premium Chocolate** was included as it shows a positive net profit of $22.00, contributing to category diversity while still maintaining a strong profitability profile.

- The allocations were determined to avoid slow-moving inventory and focus on items that maximize net profit. Products with negative net profits or high diversity penalties were excluded from allocation to ensure the shelf is optimized for profitability and product variety.

This strategy effectively balances the need for high-margin products while promoting private labels and maintaining category diversity, all aimed at maximizing overall net profit.